# 03 — Transfer Learning

Train ResNet50, VGG16, and MobileNetV2 with 2-phase training:
1. **Phase 1:** Freeze backbone, train classification head (lr=1e-3)
2. **Phase 2:** Unfreeze last layers, fine-tune end-to-end (lr=1e-5)

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

# CUDA setup — must be before TF import
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/home/yuujin/school/cv-final-project/.cuda"

import tensorflow as tf
print(f"TensorFlow {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
from src.data_loader import load_and_split_data, create_dataset, get_class_weights
from src.models import build_transfer_model
from src.train import get_callbacks, train_model, fine_tune_model
from src.evaluate import (
    evaluate_model, plot_confusion_matrix, plot_training_curves, merge_histories,
)

DATA_DIR = os.path.join("..", "data", "raw", "standardized_256")

## Load Data

In [ ]:
train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names = \
    load_and_split_data(DATA_DIR)

num_classes = len(class_names)

train_ds = create_dataset(train_paths, train_labels, augment=True)
val_ds = create_dataset(val_paths, val_labels, augment=False, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, augment=False, shuffle=False)

class_weights = get_class_weights(train_labels)

print(f"Classes ({num_classes}): {class_names}")
print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

## Training Configuration

| Model | Unfreeze Layers | Phase 1 Epochs | Phase 2 Epochs | Batch Size |
|-------|----------------|----------------|----------------|------------|
| ResNet50 | 30 | 10 | 20 | 32 |
| VGG16 | 8 | 10 | 20 | 16 (VRAM) |
| MobileNetV2 | 30 | 10 | 20 | 32 |

In [ ]:
CONFIGS = {
    "resnet50":     {"unfreeze": 30, "batch_size": 32},
    "vgg16":        {"unfreeze": 8,  "batch_size": 16},  # lower batch for 6GB VRAM
    "mobilenetv2":  {"unfreeze": 30, "batch_size": 32},
}

PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 20

all_results = {}

## ResNet50

In [ ]:
tf.keras.backend.clear_session()

name = "resnet50"
cfg = CONFIGS[name]

# Rebuild datasets if batch size differs
if cfg["batch_size"] != 32:
    t_ds = create_dataset(train_paths, train_labels, augment=True, batch_size=cfg["batch_size"])
    v_ds = create_dataset(val_paths, val_labels, augment=False, shuffle=False, batch_size=cfg["batch_size"])
else:
    t_ds, v_ds = train_ds, val_ds

model, base_model = build_transfer_model(name, num_classes)
model.summary(show_trainable=True)

# Phase 1: Train head
print("\n--- Phase 1: Training classification head ---")
callbacks = get_callbacks(name)
h1 = train_model(model, t_ds, v_ds, PHASE1_EPOCHS, class_weights, callbacks, lr=1e-3)

# Phase 2: Fine-tune
print("\n--- Phase 2: Fine-tuning last layers ---")
callbacks = get_callbacks(name)  # fresh callbacks
h2 = fine_tune_model(model, base_model, t_ds, v_ds, PHASE2_EPOCHS,
                     class_weights, callbacks, num_layers=cfg["unfreeze"], lr=1e-5)

# Plot combined training curves
merged = merge_histories(h1, h2)
plot_training_curves(merged, title="ResNet50",
                     save_path=os.path.join("..", "outputs", "plots", f"{name}_curves.png"))

# Evaluate
te_ds = create_dataset(test_paths, test_labels, augment=False, shuffle=False, batch_size=cfg["batch_size"])
results = evaluate_model(model, te_ds, class_names)
all_results[name] = results
print(f"\n{name} Test Accuracy: {results['accuracy']:.4f}")
print(results["report"])
plot_confusion_matrix(results["confusion_matrix"], class_names, title=f"{name} — Confusion Matrix",
                      save_path=os.path.join("..", "outputs", "plots", f"{name}_cm.png"))

## VGG16

In [ ]:
tf.keras.backend.clear_session()

name = "vgg16"
cfg = CONFIGS[name]

t_ds = create_dataset(train_paths, train_labels, augment=True, batch_size=cfg["batch_size"])
v_ds = create_dataset(val_paths, val_labels, augment=False, shuffle=False, batch_size=cfg["batch_size"])

model, base_model = build_transfer_model(name, num_classes)
model.summary(show_trainable=True)

print("\n--- Phase 1: Training classification head ---")
callbacks = get_callbacks(name)
h1 = train_model(model, t_ds, v_ds, PHASE1_EPOCHS, class_weights, callbacks, lr=1e-3)

print("\n--- Phase 2: Fine-tuning last layers ---")
callbacks = get_callbacks(name)
h2 = fine_tune_model(model, base_model, t_ds, v_ds, PHASE2_EPOCHS,
                     class_weights, callbacks, num_layers=cfg["unfreeze"], lr=1e-5)

merged = merge_histories(h1, h2)
plot_training_curves(merged, title="VGG16",
                     save_path=os.path.join("..", "outputs", "plots", f"{name}_curves.png"))

te_ds = create_dataset(test_paths, test_labels, augment=False, shuffle=False, batch_size=cfg["batch_size"])
results = evaluate_model(model, te_ds, class_names)
all_results[name] = results
print(f"\n{name} Test Accuracy: {results['accuracy']:.4f}")
print(results["report"])
plot_confusion_matrix(results["confusion_matrix"], class_names, title=f"{name} — Confusion Matrix",
                      save_path=os.path.join("..", "outputs", "plots", f"{name}_cm.png"))

## MobileNetV2

In [ ]:
tf.keras.backend.clear_session()

name = "mobilenetv2"
cfg = CONFIGS[name]

if cfg["batch_size"] != 32:
    t_ds = create_dataset(train_paths, train_labels, augment=True, batch_size=cfg["batch_size"])
    v_ds = create_dataset(val_paths, val_labels, augment=False, shuffle=False, batch_size=cfg["batch_size"])
else:
    t_ds, v_ds = train_ds, val_ds

model, base_model = build_transfer_model(name, num_classes)
model.summary(show_trainable=True)

print("\n--- Phase 1: Training classification head ---")
callbacks = get_callbacks(name)
h1 = train_model(model, t_ds, v_ds, PHASE1_EPOCHS, class_weights, callbacks, lr=1e-3)

print("\n--- Phase 2: Fine-tuning last layers ---")
callbacks = get_callbacks(name)
h2 = fine_tune_model(model, base_model, t_ds, v_ds, PHASE2_EPOCHS,
                     class_weights, callbacks, num_layers=cfg["unfreeze"], lr=1e-5)

merged = merge_histories(h1, h2)
plot_training_curves(merged, title="MobileNetV2",
                     save_path=os.path.join("..", "outputs", "plots", f"{name}_curves.png"))

te_ds = create_dataset(test_paths, test_labels, augment=False, shuffle=False, batch_size=cfg["batch_size"])
results = evaluate_model(model, te_ds, class_names)
all_results[name] = results
print(f"\n{name} Test Accuracy: {results['accuracy']:.4f}")
print(results["report"])
plot_confusion_matrix(results["confusion_matrix"], class_names, title=f"{name} — Confusion Matrix",
                      save_path=os.path.join("..", "outputs", "plots", f"{name}_cm.png"))

## Quick Comparison

In [ ]:
from src.evaluate import plot_model_comparison

print("\n=== Transfer Learning Results ===")
for name, res in all_results.items():
    print(f"  {name}: {res['accuracy']:.4f}")

plot_model_comparison(all_results,
                      save_path=os.path.join("..", "outputs", "plots", "transfer_comparison.png"))